In [1]:
import pandas as pd
import torch
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
DATA_DIR = "../data"

bars_seen_train         = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
bars_unseen_train       = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
bars_seen_public_test   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
bars_seen_private_test  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

headlines_seen_train        = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"), engine="fastparquet")
headlines_unseen_train      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_unseen_train.parquet"), engine="fastparquet")
headlines_seen_public_test  = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"), engine="fastparquet")
headlines_seen_private_test = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")

print("bars_seen_train:",         bars_seen_train.shape)
print("bars_unseen_train:",       bars_unseen_train.shape)
print("bars_seen_public_test:",   bars_seen_public_test.shape)
print("bars_seen_private_test:",  bars_seen_private_test.shape)
print("headlines_seen_train:",        headlines_seen_train.shape)
print("headlines_unseen_train:",      headlines_unseen_train.shape)
print("headlines_seen_public_test:",  headlines_seen_public_test.shape)
print("headlines_seen_private_test:", headlines_seen_private_test.shape)

bars_seen_train: (50000, 6)
bars_unseen_train: (50000, 6)
bars_seen_public_test: (500000, 6)
bars_seen_private_test: (500000, 6)
headlines_seen_train: (9740, 3)
headlines_unseen_train: (7631, 3)
headlines_seen_public_test: (99308, 3)
headlines_seen_private_test: (99148, 3)


In [6]:
def build_company_session_logs(df):
    df['company'] = df['headline'].str.split().str[:2].str.join(' ')
    # Group by session and company, then aggregate headlines into a list
    grouped_list_df = df.groupby(['session', 'company'])['headline'].apply(list).reset_index()

    # Rename the column for clarity
    grouped_list_df.rename(columns={'headline': 'all_headlines'}, inplace=True)
    
    return grouped_list_df


In [7]:
headlines_per_company_session = build_company_session_logs(headlines_seen_train)

In [8]:
headlines_per_company_session.head(10)

,session,company,all_headlines
0,0,Calvis Sciences,[Calvis Sciences secures $650M contract with a...
1,0,Orevex Renewables,[Orevex Renewables secures $500M contract with...
2,0,Relvos Biosciences,[Relvos Biosciences opens new office in Southe...
3,0,Yorvov Pharmaceuticals,[Yorvov Pharmaceuticals secures $180M contract...
4,1,Halvav Brands,[Halvav Brands sees mixed results in wireless ...
5,1,Jorvis Fuels,[Jorvis Fuels reports record quarterly revenue...
6,1,Strynal Industries,[Strynal Industries reports 15% decline in ope...
7,1,Zrovex Industries,[Zrovex Industries loses key contract in Asia ...
8,2,Kelvik Power,[Kelvik Power secures $800M contract with a le...
9,2,Zelval Energy,[Zelval Energy to host investor day focused on...


In [10]:
# Print the first 10 companies and their event sequences
print("--- Sample of Event Sequences ---")

for index, row in headlines_per_company_session.head(10).iterrows():
    # .ljust(25) adds spaces so the arrows all line up perfectly
    company_name = row['company'].ljust(25)
    sequence = row['all_headlines']
    
    print(f"Session {row['session']} | {company_name} -> {sequence}")

--- Sample of Event Sequences ---
Session 0 | Calvis Sciences           -> ['Calvis Sciences secures $650M contract with a leading cloud platform', 'Calvis Sciences expands operations into Asia Pacific markets']
Session 0 | Orevex Renewables         -> ['Orevex Renewables secures $500M contract with a global retailer', 'Orevex Renewables secures $650M contract with a major logistics provider', 'Orevex Renewables faces class action over automated logistics service disruption']
Session 0 | Relvos Biosciences        -> ['Relvos Biosciences opens new office in Southeast Asia', 'Relvos Biosciences names new head of precision manufacturing division', 'Relvos Biosciences reports 3% decline in operating income', 'Relvos Biosciences secures $320M contract with a leading cloud platform']
Session 0 | Yorvov Pharmaceuticals    -> ['Yorvov Pharmaceuticals secures $180M contract with a national infrastructure agency', 'Yorvov Pharmaceuticals sees mixed results in supply chain optimization pilot prog

In [126]:
def replace_headlines_with_keywords(headlines):
    new_headlines = []
    magic_words = [ "secures",
                    "expands",
                    "decline in operating income",
                    "improvement",
                    "opens",
                    "class action",
                    "loses key contract",
                    "withdraws",
                    "launches",
                    "recalls",
                    "misses",
                    "buyback program",
                    "droped",
                    "rising costs",
                    "mixed results",
                    "drop",
                    "breakthrough",
                    "acquisition",
                    "new head",
                    "strong demand",
                    "capital expenditure",
                    "scheduled maintenance",
                    "disruptions",
                    "excellence",
                    "facility upgrade",
                    "record quarterly revenue",
                    "explores strategic alternatives",
                    "routine patent applications",
                    "regulatory review",
                    "delays product launch",
                    "restructuring",
                    "revises long-term strategy",
                    "regulatory milestone",
                    "regulatory approval",
                    "host investor day",
                    "addresses investor concerns",
                    "confirms participation",
                    "enters joint venture",
                    "signs multi-year partnership",
                    "steps down unexpectedly",
                    "present at",
                    "lowers full-year guidance",
                    "publishes annual",
                    "schedules annual shareholder",
                    "major strategic initiative",
                    "potential merger",
                    "beats analyst expectations",
                    "raises full-year guidance",
                    "reports unexpected decline in"]
    
    ambiguous_headlines = []
    for headline in headlines:
        matches = 0
        for word in magic_words:
            if word in headline.lower():
                new_headlines.append(word)
                matches += 1
        if matches == 0:
            new_headlines.append(headline)    
        if matches > 1:
            ambiguous_headlines.append(headline)
            
    if len(ambiguous_headlines) != 0:
        print(f"Ambigous headlines:{ambiguous_headlines}")
    
    return new_headlines


In [127]:
headlines_per_company_session['encoded_keywords'] = headlines_per_company_session['all_headlines'].apply(replace_headlines_with_keywords)

In [128]:
master_long_headlines = []

# Assuming the column with the lists of text is named 'all_headlines'
for headline_list in headlines_per_company_session['encoded_keywords']:
    for text in headline_list:
        # Check if the text has more than 1 word
        if len(str(text).split()) > 4:
            master_long_headlines.append(text)

print(len(master_long_headlines))
for long_head in master_long_headlines:
    print(long_head) # Preview the first 5

0


In [129]:
unique_tuples = headlines_per_company_session['encoded_keywords'].apply(tuple).unique()

# 2. Convert them back to lists for easy reading
all_unique_patterns = [list(pattern) for pattern in unique_tuples]

print(f"Total unique patterns found: {len(all_unique_patterns)}\n")

# Print them all out
for pattern in all_unique_patterns:
    print(pattern)

Total unique patterns found: 2458

['secures', 'expands']
['secures', 'secures', 'class action']
['opens', 'new head', 'decline in operating income', 'secures']
['secures', 'mixed results']
['mixed results', 'rising costs']
['record quarterly revenue', 'secures', 'improvement', 'secures', 'new head', 'recalls']
['decline in operating income', 'acquisition']
['loses key contract', 'strong demand', 'acquisition', 'class action', 'rising costs', 'drop']
['secures', 'withdraws', 'secures', 'launches', 'recalls', 'misses']
['host investor day', 'misses']
['withdraws', 'scheduled maintenance']
['explores strategic alternatives', 'secures', 'strong demand']
['facility upgrade', 'steps down unexpectedly', 'routine patent applications', 'loses key contract']
['drop', 'revises long-term strategy']
['withdraws', 'acquisition']
['new head', 'buyback program']
['secures', 'launches']
['decline in operating income', 'buyback program', 'routine patent applications', 'record quarterly revenue']
['drop

In [132]:
frequency_df = headlines_per_company_session['encoded_keywords'].apply(tuple).value_counts().reset_index()
frequency_df.columns = ['encoded_keywords', 'frequency']

# 3. Convert the tuples back to lists for a cleaner look
frequency_df['encoded_keywords'] = frequency_df['encoded_keywords'].apply(list)

In [135]:
frequency_df[frequency_df["frequency"]>5]

,encoded_keywords,frequency
0,[secures],132
1,[acquisition],45
2,[revises long-term strategy],33
3,[scheduled maintenance],31
4,[mixed results],27
5,[regulatory review],24
6,[class action],23
7,[excellence],23
8,[delays product launch],23
9,"[secures, secures]",22
